# Method Draft: Embodiment-Conditioned Relational Graph Policy

本文档给出当前方法设计的 paper 风格初稿，目标是为后续实现与论文写作提供统一的技术参照。当前草稿面向 **hand embodiment generalization** 的手内旋转任务，动作空间保持为 **joint-space delta target**。方法主线基于本轮讨论形成的共识：

1. **sample-wise embodiment conditioning**，而不是 URDF 库检索；
2. **joint-level tokenization**，而不是 finger-level action abstraction；
3. **Residual Cross-Attention** 负责 state–structure alignment；
4. **Residual Relational Self-Attention** 负责结构关系消息传递；
5. **local main path + relational residual head** 保证低层控制精度与全局协调兼顾；
6. 时间维度不采用 temporal transformer，而采用 **feature-stacked short history** 或图外轻量 temporal encoder。

从论文叙事上，本方法可以概括为：

> We learn a shared joint-space manipulation policy that conditions on the current hand embodiment through structured static tokens and performs relation-aware message passing over the embodiment graph. The key design is to separate **embodiment alignment** from **relational coordination**, while preserving a direct local control path for stable high-frequency actuation.

以下章节默认以当前较完整的版本展开：`Residual Cross-Attn + Residual Relational Self-Attn + Local Bypass`。
## 3.1 Problem Setup and Notation

我们考虑一个具有可变构型的灵巧手实施体（embodiment）$e$。该手具有 $J_e$ 个可控关节，并由一个具身图
$$
G_e = (V_e, E_e)
$$
描述，其中节点对应关节或其附着 link，边对应运动学关系与几何关系。给定最近 $H$ 步的本体感受历史、当前手的静态结构描述以及关系图，策略需要输出当前时刻的 joint-space 动作：
$$
a_t \in \mathbb{R}^{J_e}.
$$

在 LEAP-style 控制设定下，我们将 $a_t$ 解释为对 joint target 的增量更新，因此控制律写作：
$$
q^{\text{target}}_{t+1} = q^{\text{target}}_t + \alpha a_t,
$$
其中 $\alpha$ 为 action scale。

### 符号约定

| 符号 | 含义 |
|---|---|
| $e$ | 当前手型（embodiment） |
| $J_e$ | 当前手型的 DoF 数 |
| $j$ | 关节索引，$j \in \{1,\dots,J_e\}$ |
| $H$ | 历史窗口长度 |
| $d_j$ | 第 $j$ 个 joint 的 dynamic token |
| $s_j$ | 第 $j$ 个 joint 的 static token |
| $s_{hand}$ | 可选的 hand-level summary token |
| $e_{ij}$ | 第 $i,j$ 个关节间的 pairwise relation feature |
| $H^{ca}$ | Cross-Attention block 输出 |
| $H^{rel}$ | Relational Self-Attention block 输出 |
| $a_j^{local}$ | local main path 为第 $j$ 个关节输出的动作 |
| $\Delta a_j^{rel}$ | relational residual branch 输出的动作修正 |

因此，方法整体目标可写为：
$$
a_t = \pi_\theta\big(\mathcal{X}^{dyn}_{t-H+1:t},\; \mathcal{X}^{stat}_e,\; G_e\big),
$$
其中 $\mathcal{X}^{dyn}_{t-H+1:t}$ 是时间变化的局部 / 全局观测历史，$\mathcal{X}^{stat}_e$ 是由 URDF / MJCF 导出的静态具身描述。

一个关键设计选择是：**我们不试图在策略内部显式检索所有训练手型的结构库，而是仅对当前样本的实施体做条件化**。因此，模型学习的是共享映射
$$
\pi_\theta: (\text{state history},\; \text{current embodiment description}) \mapsto \text{action},
$$
而不是 hand retrieval 或 nearest-neighbor matching。
## 3.2 Graph Construction and Tokenization

### 3.2.1 Dynamic observations

对第 $j$ 个 joint，在每个时间步 $t$ 我们收集局部动态观测
$$
x_{t,j}^{dyn} = [q_{t,j},\; \dot q_{t,j},\; q^{target}_{t,j},\; \phi_t],
$$
其中 $q_{t,j}$ 为当前关节位置，$\dot q_{t,j}$ 为当前关节速度，$q^{target}_{t,j}$ 为当前控制目标，$\phi_t$ 表示与周期相关的 phase 特征（如 $\sin$ / $\cos$ phase）。为了避免 temporal attention 的额外代价，我们将最近 $H$ 步的动态观测沿 feature 维堆叠：
$$
\bar x_{t,j}^{dyn} = \mathrm{Concat}(x_{t,j}^{dyn}, x_{t-1,j}^{dyn}, \dots, x_{t-H+1,j}^{dyn}).
$$
然后用一个共享的 MLP 编码为 dynamic token：
$$
d_j = \mathrm{MLP}_{dyn}(\bar x_{t,j}^{dyn}).
$$

### 3.2.2 Static embodiment observations

对于当前手型 $e$ 的第 $j$ 个关节，我们从 URDF / MJCF 中提取静态结构描述：
$$
x_j^{stat} = [q_{\min,j},\; q_{\max,j},\; a_j,\; T_{p\to j}^{rest},\; g(l_j),\; d_j^{topo}],
$$
其中：
- $q_{\min,j}, q_{\max,j}$ 为 joint limits；
- $a_j$ 为 joint axis / joint type；
- $T_{p\to j}^{rest}$ 为该关节相对父节点的 rest-pose 变换；
- $g(l_j)$ 为 attached link 的几何编码；
- $d_j^{topo}$ 为 depth、parent / child count、finger id 等拓扑离散量。

这里的 $g(l_j)$ 可以采用：
- **BPS** 编码（与 T(R,O) 思路一致）；
- 轻量几何编码，如 bbox、尺度、主轴、近似长度等。

静态 token 定义为：
$$
s_j = \mathrm{MLP}_{stat}(x_j^{stat}).
$$
此外，我们允许引入一个可选的 hand-level summary token：
$$
s_{hand} = \mathrm{Pool}(s_1, \dots, s_{J_e}),
$$
用于编码整手级的粗粒度结构信息，例如 finger count、DoF count、overall span 等。

最终 static stream 定义为：
$$
S = [s_1, s_2, \dots, s_{J_e}, s_{hand}].
$$

### 3.2.3 Variable-length handling

由于不同 hand embodiment 具有不同的 $J_e$，我们采用两种兼容的实现策略：

1. **Grouped RL setting**：若每个 env group 内只包含同一 hand embodiment，则一个 batch 内 $J_e$ 固定，无需 padding；
2. **Distillation / mixed batch setting**：若一个 batch 中混合多种 hand embodiment，则采用 padding + `src_key_padding_mask`，与 GET-Zero 一致。

因此，线性层与 attention block 本身不要求固定 DoF 数，唯一需要特殊处理的是 mixed batch 的 mask。
## 3.3 Residual Cross-Attention for Embodiment Alignment

Cross-Attention 模块的目标不是做 hand retrieval，而是做 **state-conditioned structure retrieval within the current hand**。在进入 cross-attention 之前，我们先对 dynamic token 做一层局部 conditioning：

### 3.3.1 Local conditioning

我们给出两种实现：

**FiLM 版本**：
$$
\hat d_j = \gamma(s_j) \odot d_j + \beta(s_j),
$$
其中 $\gamma(\cdot), \beta(\cdot)$ 为由 static token 预测的 modulation 参数。

**Concat 版本**：
$$
\hat d_j = \mathrm{MLP}_{cond}([d_j, s_j]).
$$

默认草稿采用 **FiLM**，因为它能更直接保留原始 local dynamics，同时只做 feature-wise modulation。

### 3.3.2 Sample-wise cross-attention

然后，我们在当前样本内部计算 structured cross-attention：
$$
\hat D = [\hat d_1, \dots, \hat d_{J_e}],
$$
$$
H^{ca} = \hat D + \mathrm{CrossAttn}(\mathrm{LN}(\hat D), \mathrm{LN}(S), \mathrm{LN}(S)),
$$
$$
H^{ca} = H^{ca} + \mathrm{FFN}(\mathrm{LN}(H^{ca})).
$$

这里采用标准的 residual + FFN 结构，而不是裸的 attention layer。对第 $i$ 个 joint，其 cross-attention 形式为：
$$
\mathrm{CrossAttn}(\hat d_i, S, S) = \sum_{k} \alpha_{ik}^{ca} W_V s_k,
$$
$$
\alpha_{ik}^{ca} \propto \exp\left(\frac{(W_Q \hat d_i)(W_K s_k)^T}{\sqrt d}\right).
$$

### 3.3.3 Design motivation

该模块的动机是：
- 将当前 joint 的动态状态与**当前手的结构化 embodiment tokens** 对齐；
- 允许某个 joint token 读取其他 joint 的结构信息，而不是只读取自身局部静态属性；
- 在进入 relation block 之前，先显式完成 state–structure alignment。

如果后续实验发现 cross-attention 收益有限，可将该模块退化为纯 FiLM / concat conditioning，而不影响后续 relational block。
## 3.4 Relational Self-Attention with Graph Bias

在完成 embodiment alignment 之后，我们使用 relation-aware self-attention 在 joint graph 上进行结构化通信。与 GET-Zero 仅在 attention score 中加入离散 graph bias 不同，我们同时引入：

1. **离散拓扑 bias**：SPD、parent、child 等；
2. **连续几何关系**：rest-pose 下的相对 SE(3)、link-pair geometry 等；
3. **edge-conditioned value**：让消息内容本身受 $e_{ij}$ 调制。

### 3.4.1 Pairwise relation feature

对每一对关节 $(i,j)$，构造关系特征：
$$
e_{ij} = [\phi^{SPD}(i,j),\; \phi^P(i,j),\; \phi^C(i,j),\; \Delta T_{ij}^{rest},\; \Delta g_{ij}],
$$
其中：
- $\phi^{SPD}(i,j)$：无向最短路径距离；
- $\phi^P(i,j), \phi^C(i,j)$：有向 parent / child 关系；
- $\Delta T_{ij}^{rest}$：rest pose 下的相对变换；
- $\Delta g_{ij}$：由 link geometry 派生的 pairwise 关系描述。

### 3.4.2 Relational self-attention

在第 $\ell$ 层，记输入为 $h_i^{(\ell)}$。我们定义：
$$
Q_i = W_Q h_i^{(\ell)}, \qquad K_j = W_K h_j^{(\ell)},
$$
$$
\alpha_{ij}^{(\ell)} \propto \exp\left(\frac{Q_i K_j^T}{\sqrt d} + b_{ij}^{topo} + b_{ij}^{geom}\right),
$$
$$
\tilde V_{ij}^{(\ell)} = h_\psi\big(h_i^{(\ell)}, h_j^{(\ell)}, e_{ij}\big),
$$
$$
\hat h_i^{(\ell)} = \sum_j \alpha_{ij}^{(\ell)} \tilde V_{ij}^{(\ell)}.
$$

然后采用 residual + FFN：
$$
H^{rel} = H^{ca} + \mathrm{RelSelfAttn}(\mathrm{LN}(H^{ca}), e),
$$
$$
H^{rel} = H^{rel} + \mathrm{FFN}(\mathrm{LN}(H^{rel})).
$$

### 3.4.3 Why relation-conditioned value

该模块的核心观点是：**连续几何关系不应只影响“看谁更多”，还应影响“传什么消息”**。这与 TRO-Grasp 中 OR / RR attention 的设计是一致的，但这里我们将其简化为适合在线 joint-space control 的 homogeneous joint graph 版本。

### 3.4.4 Complexity

若当前手具有 $J_e$ 个 joint，则每层的主要复杂度为：
$$
\mathcal O(J_e d^2 + J_e^2 d).
$$
对于 LEAP-style 16-DoF hands，$J_e^2$ 的量级较小，因此这一模块对 `>20Hz` 的 online control 是可接受的。
## 3.5 Joint-Space Action Head and Auxiliary Objectives

### 3.5.1 Local main path + relational residual head

为了避免低层 proprioceptive 信息在多层 attention 后被过度混合，我们显式保留一条 local main path。对第 $j$ 个关节：

$$
a_j^{local} = \pi_{local}(\hat d_j),
$$
$$
\Delta a_j^{rel} = \pi_{rel}([\hat d_j, h_j^{rel}]),
$$
$$
a_j = a_j^{local} + \Delta a_j^{rel}.
$$

可选地，可增加 gate：
$$
a_j = a_j^{local} + \lambda_j \odot \Delta a_j^{rel},
$$
$$
\lambda_j = \sigma(W h_j^{rel}).
$$

默认草稿中，**无 gate 的 residual sum** 作为主实现，gated residual 作为可选 ablation。

### 3.5.2 Auxiliary objectives

为了让最终表示更结构敏感，我们保留 GET-Zero 风格的 self-modeling 思路，默认采用 forward kinematics 预测作为辅助目标：
$$
\hat p_j = f_{fk}(h_j^{rel}),
$$
$$
L_{fk} = \frac{1}{J_e} \sum_{j=1}^{J_e} \lVert \hat p_j - p_j^{gt} \rVert_2^2.
$$

训练目标根据训练范式不同而变化：

**Behavior cloning / distillation**：
$$
L_{act}^{BC} = \frac{1}{J_e} \sum_{j=1}^{J_e} \lVert a_j - a_j^* \rVert_1
\quad \text{or} \quad
\lVert a_j - a_j^* \rVert_2^2.
$$

**Actor-critic / PPO**：
$$
L_{act}^{RL} = L_{PPO} + c_v L_{value} - c_e H(\pi).
$$

统一写法为：
$$
L = L_{act} + \lambda_{fk} L_{fk} + \lambda_{reg} L_{reg},
$$
其中 $L_{reg}$ 可用于约束 gate、残差幅值或参数范数。

### 3.5.3 Why this head design

这种 local main path + relational residual 的分工有两个优势：

1. low-level proprioceptive control 能直接映射到动作，不必强迫 relation module 重建所有局部信息；
2. relation branch 的职责被压缩成“修正量 / 协调量”，更符合它的功能定位。
## 3.6 Forward Pass Pseudocode

下面给出一版实现导向的前向传播伪代码：

1. **Parse embodiment**
   - 从当前 hand 的 URDF / MJCF 中提取每个 joint 的静态字段；
   - 为每个 joint 构造 static token $s_j$；
   - 可选地构造 $s_{hand}$。

2. **Build dynamic tokens**
   - 从最近 $H$ 步的 proprio / target / phase 中构造 $\bar x_{t,j}^{dyn}$；
   - 通过 `MLP_dyn` 得到 $d_j$。

3. **Local conditioning**
   - 用 FiLM 或 concat 将 $s_j$ 融入 $d_j$，得到 $\hat d_j$。

4. **Residual cross-attention block**
   - 输入：$\hat D$ 与 $S$；
   - 输出：$H^{ca}$。

5. **Build relation edges**
   - 根据当前 embodiment 的 graph 与静态几何，构造 $e_{ij}$；
   - 若需要动态 relation，可在后续版本中额外加入实时接触 / pose 派生项。

6. **Residual relational self-attention block**
   - 输入：$H^{ca}$ 与 $e_{ij}$；
   - 输出：$H^{rel}$。

7. **Action head**
   - local path：$a_j^{local} = \pi_{local}(\hat d_j)$；
   - relational path：$\Delta a_j^{rel} = \pi_{rel}([\hat d_j, h_j^{rel}])$；
   - 输出：$a_j = a_j^{local} + \Delta a_j^{rel}$。

8. **Auxiliary head (optional)**
   - 预测 joint FK 位置；
   - 计算 $L_{fk}$。

该伪代码对应的模块化实现边界很清楚：
- `StaticEncoder`
- `DynamicEncoder`
- `LocalConditioning`
- `ResidualCrossAttentionBlock`
- `RelationBuilder`
- `ResidualRelationalAttentionBlock`
- `LocalHead`
- `RelationalHead`
- `FKHead`
## 3.7 Implementation Notes and Default Draft Choices

### 3.7.1 History handling

若目标是 `>20Hz`，默认不采用 temporal attention。当前更推荐的两个实现是：

1. **feature-stack history**（默认）
   - 与 GET-Zero 一致，简单稳定；
   - 直接将最近 $H$ 步观测沿 feature 维堆叠进每个 joint token。

2. **graph-external temporal encoder**（备选）
   - 对每个 joint 使用小型 GRU / TCN；
   - 再将其输出送入 graph module；
   - 时间建模与空间关系解耦。

### 3.7.2 Padding strategy

- **Grouped RL**：同一 env group 内 DoF 数相同，无需 padding；
- **Mixed-batch distillation**：使用 padding + mask，与 GET-Zero 对齐。

### 3.7.3 Default draft training protocol

当前方法草稿暂定采用以下默认训练顺序：

1. **teacher phase**：对若干 seen embodiments 训练 teacher policies；
2. **distillation phase**：用统一的 embodiment-conditioned relational policy 蒸馏 teacher 行为；
3. **optional fine-tuning**：在多手 env groups 上做少量 RL fine-tuning。

这样做的理由是：
- 直接 multi-embodiment end-to-end RL 的不稳定性仍未完全澄清；
- distillation 更接近 GET-Zero 的成功先例；
- 同时保留后续转向 RL fine-tuning 的空间。

### 3.7.4 Recommended ablations

建议至少包含以下消融：

| Ablation | 目的 |
|---|---|
| 去掉 Cross-Attn，仅保留 FiLM / concat | 验证显式 state–structure alignment 的价值 |
| 将 edge-conditioned value 退化为 bias-only | 对比方案 A 与方案 B |
| 去掉 local bypass | 验证 local main path 的必要性 |
| 去掉 FK self-modeling | 验证结构辅助监督是否有益 |
| 不同 history 方案（single-step / stack / GRU） | 验证时间建模选择 |

### 3.7.5 Complexity and control-rate note

由于当前手型的 joint 数量有限（例如 LEAP Hand 约 16 DoF），空间 attention 的复杂度在实践中较小。部署到真实硬件时，更大的频率风险通常来自：
- hardware I/O；
- sensor processing；
- Python control loop；
- tokenizer / observation construction。

因此，方法实现时应优先遵守两个原则：

1. **时间模块放图外**；
2. **关系模块只在 joint graph 上做，不引入大规模 object patch graph。**

---

**当前草稿状态**：
- 架构主干已基本明确；
- 仍待最终拍板的，是 `history 方案`、`training protocol`、`static token 字段表`、`relation edge 字段表` 和 `policy head 具体实现变体`。